# SageMaker End-to-End ML Project - Experimentation

This notebook walks through the full ML lifecycle using AWS SageMaker from VS Code.

**Use Case**: Customer Churn Prediction (binary classification)

## Steps:
1. Setup & Configuration
2. Data Loading & Exploration
3. Data Preprocessing
4. Model Training on SageMaker
5. Model Evaluation
6. Deploy Model to Endpoint
7. Test Predictions
8. Cleanup

## Step 1: Setup & Configuration

Before running this notebook:
- Run `aws configure` in your terminal to set up credentials
- Ensure you have an IAM role with SageMaker permissions
- Install requirements: `pip install -r ../requirements.txt`

In [ ]:
import boto3
import sagemaker
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path

# Load project config
config_path = Path('../config/pipeline_params.json')
with open(config_path) as f:
    config = json.load(f)

print(f"Project: {config['project_name']}")
print(f"Region: {config['region']}")

In [ ]:
# Initialize SageMaker session
# NOTE: When running from VS Code (not SageMaker Studio), you need to specify
# the role ARN directly since get_execution_role() only works inside SageMaker.

region = config['region']
boto_session = boto3.Session(region_name=region)
sm_client = boto_session.client('sagemaker')
session = sagemaker.Session(boto_session=boto_session)

# IMPORTANT: Replace this with your actual SageMaker execution role ARN
# You can find this in IAM console > Roles > search for 'SageMaker'
ROLE_ARN = 'arn:aws:iam::YOUR_ACCOUNT_ID:role/YOUR_SAGEMAKER_ROLE'

bucket = session.default_bucket()
prefix = config['project_name']

print(f"SageMaker SDK version: {sagemaker.__version__}")
print(f"S3 Bucket: {bucket}")
print(f"Prefix: {prefix}")
print(f"Region: {region}")

## Step 2: Data Loading & Exploration

We'll use a synthetic churn dataset. In a real project, you'd pull from your data lake or database.

In [ ]:
# Generate synthetic churn data for demonstration
# In production, replace this with your actual data source
np.random.seed(42)
n_samples = 5000

data = pd.DataFrame({
    'customer_id': range(1, n_samples + 1),
    'tenure_months': np.random.randint(1, 72, n_samples),
    'monthly_charges': np.random.uniform(20, 100, n_samples).round(2),
    'total_charges': np.random.uniform(100, 7000, n_samples).round(2),
    'contract_type': np.random.choice(['Month-to-month', 'One year', 'Two year'], n_samples, p=[0.5, 0.3, 0.2]),
    'payment_method': np.random.choice(['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'], n_samples),
    'internet_service': np.random.choice(['DSL', 'Fiber optic', 'No'], n_samples, p=[0.35, 0.45, 0.2]),
    'online_security': np.random.choice(['Yes', 'No'], n_samples),
    'tech_support': np.random.choice(['Yes', 'No'], n_samples),
    'num_support_tickets': np.random.poisson(2, n_samples),
})

# Create target variable with some logic
churn_probability = (
    0.3 * (data['contract_type'] == 'Month-to-month').astype(float) +
    0.2 * (data['tenure_months'] < 12).astype(float) +
    0.15 * (data['monthly_charges'] > 70).astype(float) +
    0.1 * (data['num_support_tickets'] > 3).astype(float) +
    np.random.uniform(0, 0.3, n_samples)
)
data['churn'] = (churn_probability > 0.5).astype(int)

print(f"Dataset shape: {data.shape}")
print(f"\nChurn distribution:")
print(data['churn'].value_counts(normalize=True))
data.head()

In [ ]:
# Quick EDA
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Churn by contract type
sns.countplot(data=data, x='contract_type', hue='churn', ax=axes[0, 0])
axes[0, 0].set_title('Churn by Contract Type')

# Tenure distribution
sns.histplot(data=data, x='tenure_months', hue='churn', kde=True, ax=axes[0, 1])
axes[0, 1].set_title('Tenure Distribution by Churn')

# Monthly charges
sns.boxplot(data=data, x='churn', y='monthly_charges', ax=axes[1, 0])
axes[1, 0].set_title('Monthly Charges by Churn')

# Support tickets
sns.countplot(data=data, x='num_support_tickets', hue='churn', ax=axes[1, 1])
axes[1, 1].set_title('Support Tickets by Churn')

plt.tight_layout()
plt.show()

## Step 3: Data Preprocessing

We'll preprocess locally first, then move this logic into a SageMaker Processing job.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Drop customer_id (not a feature)
df = data.drop('customer_id', axis=1)

# Encode categorical variables
categorical_cols = ['contract_type', 'payment_method', 'internet_service', 'online_security', 'tech_support']
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# Split features and target
# IMPORTANT: For XGBoost on SageMaker, target column must be FIRST
target = 'churn'
feature_cols = [col for col in df.columns if col != target]
df_final = df[[target] + feature_cols]

# Train/validation/test split
train_df, temp_df = train_test_split(df_final, test_size=0.3, random_state=42, stratify=df_final[target])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df[target])

print(f"Train: {train_df.shape}")
print(f"Validation: {val_df.shape}")
print(f"Test: {test_df.shape}")

In [ ]:
# Save processed data locally
data_dir = Path('../data')
train_df.to_csv(data_dir / 'train.csv', index=False, header=False)
val_df.to_csv(data_dir / 'validation.csv', index=False, header=False)
test_df.to_csv(data_dir / 'test.csv', index=False, header=False)

print("Data saved locally.")
print(f"  Train: {data_dir / 'train.csv'}")
print(f"  Validation: {data_dir / 'validation.csv'}")
print(f"  Test: {data_dir / 'test.csv'}")

In [ ]:
# Upload to S3
train_s3_uri = session.upload_data(
    path=str(data_dir / 'train.csv'),
    bucket=bucket,
    key_prefix=f'{prefix}/data/train'
)
val_s3_uri = session.upload_data(
    path=str(data_dir / 'validation.csv'),
    bucket=bucket,
    key_prefix=f'{prefix}/data/validation'
)
test_s3_uri = session.upload_data(
    path=str(data_dir / 'test.csv'),
    bucket=bucket,
    key_prefix=f'{prefix}/data/test'
)

print(f"Train uploaded to: {train_s3_uri}")
print(f"Validation uploaded to: {val_s3_uri}")
print(f"Test uploaded to: {test_s3_uri}")

## Step 4: Model Training on SageMaker

Now we launch a training job on SageMaker managed infrastructure.
This runs on a separate ML instance — your local machine just submits the job.

In [ ]:
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput

# Get the XGBoost container image URI
xgb_image_uri = sagemaker.image_uris.retrieve(
    framework='xgboost',
    region=region,
    version='1.7-1'
)
print(f"XGBoost image: {xgb_image_uri}")

# Define the estimator
xgb_estimator = Estimator(
    image_uri=xgb_image_uri,
    role=ROLE_ARN,
    instance_count=config['training_instance_count'],
    instance_type=config['training_instance_type'],
    output_path=f's3://{bucket}/{prefix}/models',
    sagemaker_session=session,
    base_job_name='churn-xgboost'
)

# Set hyperparameters from config
xgb_estimator.set_hyperparameters(**config['hyperparameters'])

print("Estimator configured. Ready to train.")

In [ ]:
# Launch training job
# This will take 3-5 minutes (instance spin-up + training)
xgb_estimator.fit(
    inputs={
        'train': TrainingInput(s3_data=train_s3_uri, content_type='text/csv'),
        'validation': TrainingInput(s3_data=val_s3_uri, content_type='text/csv')
    },
    wait=True,  # Set to False to run async
    logs='All'
)

print(f"\nTraining complete!")
print(f"Model artifact: {xgb_estimator.model_data}")

## Step 5: Model Evaluation

We'll use SageMaker Batch Transform to get predictions on the test set, then evaluate.

In [ ]:
# Create a transformer for batch predictions
transformer = xgb_estimator.transformer(
    instance_count=1,
    instance_type='ml.m5.large',
    output_path=f's3://{bucket}/{prefix}/batch-predictions'
)

# Run batch transform on test data (without the target column)
# First, create test data without target
test_no_target = test_df.iloc[:, 1:]  # Remove first column (target)
test_no_target_path = str(data_dir / 'test_no_target.csv')
test_no_target.to_csv(test_no_target_path, index=False, header=False)

test_no_target_s3 = session.upload_data(
    path=test_no_target_path,
    bucket=bucket,
    key_prefix=f'{prefix}/data/test_no_target'
)

transformer.transform(
    data=test_no_target_s3,
    content_type='text/csv',
    split_type='Line'
)
transformer.wait()
print("Batch transform complete!")

In [ ]:
# Download and evaluate predictions
import io
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# Get predictions from S3
s3 = boto3.client('s3', region_name=region)
prediction_key = f'{prefix}/batch-predictions/test_no_target.csv.out'

response = s3.get_object(Bucket=bucket, Key=prediction_key)
predictions_raw = response['Body'].read().decode('utf-8')
predictions = np.array([float(x) for x in predictions_raw.strip().split('\n')])

# Convert probabilities to binary predictions
y_true = test_df.iloc[:, 0].values
y_pred = (predictions > 0.5).astype(int)

# Metrics
print("=" * 50)
print("MODEL EVALUATION RESULTS")
print("=" * 50)
print(f"\nROC AUC Score: {roc_auc_score(y_true, predictions):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=['No Churn', 'Churn']))
print(f"Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

## Step 6: Deploy Model to Real-Time Endpoint

Deploy the trained model for real-time inference.

In [ ]:
# Deploy to a real-time endpoint
predictor = xgb_estimator.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large',
    serializer=sagemaker.serializers.CSVSerializer(),
    deserializer=sagemaker.deserializers.CSVDeserializer()
)

print(f"Endpoint deployed: {predictor.endpoint_name}")

In [ ]:
# Test the endpoint with sample data
sample = test_no_target.iloc[:5].values

for i, row in enumerate(sample):
    result = predictor.predict(row.tolist())
    prob = float(result[0][0])
    actual = int(y_true[i])
    print(f"Sample {i+1}: Predicted={prob:.3f} ({'Churn' if prob > 0.5 else 'No Churn'}) | Actual={'Churn' if actual == 1 else 'No Churn'}")

## Step 7: Cleanup

**IMPORTANT**: Delete the endpoint when done to avoid ongoing charges!

In [ ]:
# DELETE ENDPOINT - uncomment when done testing
# predictor.delete_endpoint()
# print("Endpoint deleted.")

## Next Steps: Convert to MLOps Pipeline

Now that we've validated the approach interactively, the next step is to:

1. Extract preprocessing into `scripts/preprocess.py`
2. Extract training logic into `scripts/train.py`
3. Extract evaluation into `scripts/evaluate.py`
4. Wire everything together in `pipelines/pipeline.py`

This gives you:
- **Reproducibility**: Same pipeline runs every time
- **Automation**: Trigger on new data or schedule
- **Versioning**: Track model versions in Model Registry
- **CI/CD**: Integrate with CodePipeline or GitHub Actions